# Stock Portfolio Diversification Using Asset Clustering

Cluster assets by return behaviour to support more diversified portfolio selection.

**Portfolio category:** Financial clustering

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Multi-factor asset-return simulation

In [ ]:
sectors = ["technology", "finance", "health", "energy", "consumer", "industrial"]
assets = [f"{sector[:3].upper()}_{i + 1}" for sector in sectors for i in range(4)]
days = 756
market_factor = rng.normal(0.0003, 0.009, days)
sector_factors = {sector: rng.normal(0.0001, 0.011, days) for sector in sectors}
returns = {}
asset_sector = {}
for asset, sector in zip(assets, np.repeat(sectors, 4)):
    beta = rng.uniform(0.7, 1.3)
    sector_loading = rng.uniform(0.7, 1.2)
    returns[asset] = beta * market_factor + sector_loading * sector_factors[sector] + rng.normal(0, 0.008, days)
    asset_sector[asset] = sector
returns = pd.DataFrame(returns, index=pd.date_range("2023-01-01", periods=days, freq="B"))
display(returns.head())

## 3. Return quality and summary

In [ ]:
print("Missing cells:", int(returns.isna().sum().sum()))
summary = pd.DataFrame({
    "annual_return": returns.mean() * 252,
    "annual_volatility": returns.std() * np.sqrt(252),
    "downside_volatility": returns.clip(upper=0).std() * np.sqrt(252),
    "market_correlation": returns.corrwith(pd.Series(market_factor, index=returns.index)),
})
summary["sharpe_proxy"] = summary["annual_return"] / summary["annual_volatility"]
display(summary.round(3))

## 4. Correlation-aware asset features

In [ ]:
correlation = returns.corr()
correlation_features = correlation.add_prefix("corr_")
features = summary.join(correlation_features)
X = StandardScaler().fit_transform(features)

## 5. Hierarchical asset clustering

In [ ]:
linkage_matrix = linkage(X, method="ward")
candidate_rows = []
for k in range(3, 9):
    labels = fcluster(linkage_matrix, t=k, criterion="maxclust")
    candidate_rows.append({"k": k, "silhouette": silhouette_score(X, labels)})
scores = pd.DataFrame(candidate_rows)
best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
summary["cluster"] = fcluster(linkage_matrix, t=best_k, criterion="maxclust")
summary["sector"] = pd.Series(asset_sector)
display(scores.round(3))

## 6. Diversification diagnostics

In [ ]:
within = []
between = []
for left in assets:
    for right in assets:
        if left >= right:
            continue
        target = within if summary.loc[left, "cluster"] == summary.loc[right, "cluster"] else between
        target.append(correlation.loc[left, right])
display(pd.Series({
    "selected_clusters": best_k,
    "mean_within_cluster_correlation": np.mean(within),
    "mean_between_cluster_correlation": np.mean(between),
}).to_frame("value"))

## 7. Dendrogram

In [ ]:
plt.figure(figsize=(12, 5))
dendrogram(linkage_matrix, labels=assets, leaf_rotation=90)
plt.title("Asset similarity dendrogram")
plt.tight_layout()

## 8. Representative assets

In [ ]:
representatives = summary.sort_values(["cluster", "sharpe_proxy"], ascending=[True, False]).groupby("cluster").head(1)
display(representatives.round(3))

## 9. Key findings

Clustering can expose redundant exposures, but portfolio construction still requires risk limits, costs and investment review.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For stock portfolio diversification using asset clustering,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.